# Fused compartment updates

By default `compile(fuse_compartment_updates=True)` applies every compartment update with
**one** sparse `scatter-add` after the per-flow mass loop. The older path
(`fuse_compartment_updates=False`) scatters once or twice per named flow.
Source gathers stay per-flow (fusing them hurt `grad` via slice→pad).

Claim to check: both modes produce the same `dy`, and the fused vector field
carries fewer `scatter-add` ops in its jaxpr. The bar chart compares those
counts on a small multi-flow SIR.


In [ ]:
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import jax
import numpy as np

from summer4 import (
    EntryFlow,
    Everything,
    ExitFlow,
    FlowModel,
    Property,
    PropertyMap,
    TransitionFlow,
)


## Same `dy`, fewer scatters

Infection, recovery, death, and replacement births. Compile twice and compare
the vector field and its jaxpr.


In [ ]:
state = Property("state", ("S", "I", "R"))
pm = PropertyMap.from_property(state)
model = FlowModel(pm)
model.add_flow(TransitionFlow("inf", state["S"], state["I"], 0.1))
death = model.add_flow(ExitFlow("death", Everything(), 0.01))
model.add_flow(EntryFlow("birth", state["S"], death.sum()))
model.add_flow(TransitionFlow("rec", state["I"], state["R"], 0.05))

fused = model.compile(fuse_compartment_updates=True)
looped = model.compile(fuse_compartment_updates=False)
y = np.array([900.0, 80.0, 20.0])

dy_f = np.asarray(fused.vector_field(0.0, y, {}))
dy_l = np.asarray(looped.vector_field(0.0, y, {}))
np.testing.assert_allclose(dy_f, dy_l, rtol=0.0, atol=0.0)


def scatter_adds(cm):
    jaxpr = jax.make_jaxpr(cm.vector_field)(0.0, y, {})
    return sum(1 for e in jaxpr.jaxpr.eqns if e.primitive.name == "scatter-add")


n_fused = scatter_adds(fused)
n_looped = scatter_adds(looped)
assert n_fused < n_looped

counts = pd.DataFrame(
    {
        "mode": ["fuse_compartment_updates=True", "fuse_compartment_updates=False"],
        "scatter-add ops": [n_fused, n_looped],
    }
).set_index("mode")
fig = counts.plot.bar(
    title="Vector-field scatter-add ops (fused vs per-flow)",
    labels={"value": "scatter-add count", "mode": "compile mode"},
)
fig.show()
print(f"fused={n_fused}, looped={n_looped}, dy={dy_f}")
